## Running SQL Queries Pandas DataFrames
- You can run SQL queries in Pandas Dataframe
    - but how?

Overview
- Pandas is used for data analysis purposes, and it has the advantage of being part of the wider Python universe, making it accessible to many people. 
- SQL is known for its performance, being human-readable, and can be easily understood even by non-technical people.

**Can we find a way to combine the benefits of both Pandas and SQL statements?**
- come in `pandasql` module

First install `pandasql` using `pip install pandasql` or  `!conda install pandasql --yes`
    
**Import the libraries**


In [7]:
#pip install pandasql

In [6]:
from pandasql import sqldf
import pandas as pd

**We create a sample dataset**

In [8]:
# Create the Student Data Frame

students= { 
    'Students':["Sira","Ibrahim","Moussa","Mamadou","Nabintou"],
    'Gender':['Female','Male','Male', "Male", "Female"],
    'Age':[18, 27, 19, 22, 21],
    'Email': ["sira@info.com", "ib@info.com", "mouss@info.com", 
             "mam@info.com", "nab@info.com"]
          }
students_df = pd.DataFrame(students)

students_df

,Students,Gender,Age,Email
0,Sira,Female,18,sira@info.com
1,Ibrahim,Male,27,ib@info.com
2,Moussa,Male,19,mouss@info.com
3,Mamadou,Male,22,mam@info.com
4,Nabintou,Female,21,nab@info.com


In [9]:
# Create the Teaching Assistant Data Frame

teaching_assistant= {
    'Teacher':["Ibrahim","Nabintou","Mamadou","Fatim","Aziz"],
    'Email':['ib@info.com','nab@info.com','mam@info.com', 
             "fat@info.com", "aziz@info.com"],
    'Degree':["MSc. in Data Science", "BSc. in Statistics", 
              "BSc. Comp Sci", "MSc. Architecture", "BSc. in Accounting"],
    'Department': ["Business", "Statistics", "Comp Sci", 
             "Engineering", "Business"]
          }
teaching_assistant_df = pd.DataFrame(teaching_assistant)

teaching_assistant_df

,Teacher,Email,Degree,Department
0,Ibrahim,ib@info.com,MSc. in Data Science,Business
1,Nabintou,nab@info.com,BSc. in Statistics,Statistics
2,Mamadou,mam@info.com,BSc. Comp Sci,Comp Sci
3,Fatim,fat@info.com,MSc. Architecture,Engineering
4,Aziz,aziz@info.com,BSc. in Accounting,Business


**What we will try to do**

- column selection
- data filtering
- data aggregation
- data joining.

#### 1- Column Selection
This corresponds to selecting part or all the columns of your table/data frame. 
- It is performed with the expression `SELECT col_1, col_2,... col_X FROM tableName`

In [ ]:
students_df.head()

In [5]:
all_students = sqldf("SELECT * FROM students_df limit 5")
all_students

,Students,Gender,Age,Email
0,Sira,Female,18,sira@info.com
1,Ibrahim,Male,27,ib@info.com
2,Moussa,Male,19,mouss@info.com
3,Mamadou,Male,22,mam@info.com
4,Nabintou,Female,21,nab@info.com


**Reflection**

- `all_students` is a DataFrame, which can be helpful if you want to perform quick Pandas tasks such as column types. Below is an illustration.

In [6]:
# Check the type of all_students
print(type(all_students))

# Run Pandas Statement to show the type of the columns
print("---"*12)
print(all_students.dtypes)

<class 'pandas.core.frame.DataFrame'>
------------------------------------
Students    object
Gender      object
Age          int64
Email       object
dtype: object


### Handling complicated queries
-  when you have complicated queries, you can proceed step by step as follow:

- Define the query as a string. 
    - When doing so, make sure to use the triple quote sign """ so that you can write the query on multiple lines.
- Apply the sqldf function to the query to get the result.

**Example**

- suppose we want the student's name, their email and limit the result to the first 3.

In [ ]:
students_df['Email'].head(3)

In [7]:
# Query definition
query = """ SELECT Students, Email 
            FROM students_df 
            LIMIT 3
        """
# Query execution
name_email = sqldf(query)
name_email

,Students,Email
0,Sira,sira@info.com
1,Ibrahim,ib@info.com
2,Moussa,mouss@info.com


**2- Data Filtering**

- this requires the  `WHERE` statement to perform custom filtering on the result of the SELECT statement.

Let’s imagine that we want all the female students.

In [8]:
# Define the query
query = """SELECT * 
           FROM students_df 
           WHERE Gender = 'Female'
        """

# Execute the query
female_student = sqldf(query)
female_student

,Students,Gender,Age,Email
0,Sira,Female,18,sira@info.com
1,Nabintou,Female,21,nab@info.com


Consider a query that aims to retrieve the `Name`, `Email`, and `Degree` of all the Teaching Assistants with a Master's Degree.

In [9]:
query = """ SELECT Teacher, Email, Degree 
            FROM teaching_assistant_df 
            WHERE Degree LIKE 'MS%'
            """
ms_students = sqldf(query)
ms_students

,Teacher,Email,Degree
0,Ibrahim,ib@info.com,MSc. in Data Science
1,Fatim,fat@info.com,MSc. Architecture


**Can we do the same query above using pandas syntax?**

Yes we can:

In [10]:
cols_to_get = ['Teacher', 'Email', 'Degree']
teaching_assistant_df[teaching_assistant_df.Degree.str.startswith('MS')][cols_to_get]

,Teacher,Email,Degree
0,Ibrahim,ib@info.com,MSc. in Data Science
3,Fatim,fat@info.com,MSc. Architecture


What the pandas statement means above:

- `cols_to_get` corresponds to the columns we want in the final dataframe. It’s basically equivalent to the columns we specify when using the SELECT statement.
- `teaching_assistant_df.Degree.str.startswith('MS')` returns `True` if the Degree column starts with 'MS', False Otherwise.
    - those boolean values are transmitted to the starting teaching_assistant_df to get rows where the result is True.

#### 3- Data Aggregation
- `Aggregation` in SQL is performed with the help of aggregation functions
- the most commonly used are `COUNT` , `SUM` , `MAX`, `MIN` , and `AVG` 

**Example**

We obtain the average age of students, based on their gender.

In [12]:
query = """ SELECT AVG(Age), Gender 
            FROM students_df 
            GROUP BY Gender
        """
avg_age = sqldf(query)
avg_age

,AVG(Age),Gender
0,19.500000,Female
1,22.666667,Male


**How can you perform the above aggregation using pandas?**

Write your statement below:

#### 4- Data Joining
- The join concept becomes useful when we want to combine data from at least two tables. 

**Example**

- Suppose we want to know who is both a student and also a teaching assistant. 
    - this requires joining our two tables using an `INNER JOIN`.
    The final result contains the following information:

        - Student Name, Gender, Email, and Age from the students_df table.
        - Department from the teaching_assistant_df dataframe.

In [13]:
query = """ SELECT st.Students, st.Gender, st.Email, st.Age, tat.Department
            FROM students_df st INNER JOIN teaching_assistant_df tat 
            ON st.Email = tat.Email;
            """

result = sqldf(query)
result

,Students,Gender,Email,Age,Department
0,Ibrahim,Male,ib@info.com,27,Business
1,Mamadou,Male,mam@info.com,22,Comp Sci
2,Nabintou,Female,nab@info.com,21,Statistics


**Perform the above join using pandas**